## 📊 Telco Customer Churn Analysis

### 🧹 Data Cleaning & Understanding

#### 🎯 Objective
The goal of this notebook is to clean and prepare the dataset for analysis by handling missing values, correcting data types, and creating meaningful features.

#### ** Why Data Cleaning Matters? **
Real-world data is often messy and inconsistent. Proper cleaning ensures that the analysis is accurate, reliable, and meaningful.

> “Garbage in, garbage out” — poor data quality leads to poor insights.

In [21]:
import pandas as pd
import numpy as np
import matplotlib as plt

### 📂 Load Dataset

We begin by loading the dataset and inspecting its structure.

In [22]:
df = pd.read_csv("/Users/hp/Documents/ASTU/CSEC_Projects/DataScience/telco-churn-analysis/data/raw/Telco_Customer_Churn_Dataset.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


#### 🔍 Initial Data Inspection

Understanding the dataset structure, data types, and completeness.

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [24]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


#### ⚠️ Missing Values Analysis

Checking for missing data to ensure reliability of analysis.

In [25]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

### 🔧 Data Type Correction

The `TotalCharges` column is incorrectly stored as a string. 
We convert it to numeric for accurate calculations.

errors='coerce' converts invalid values (like blank spaces) to NaN

In [26]:
# --- Clean TotalCharges ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')


Now the invalid values are converted to missing values to maintain consistency. we can check them now

In [27]:
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [39]:
# Check BEFORE mapping
print(df['Churn'].unique())

['no' 'yes']


### 🧹 Handling Missing Values

Since missing values are minimal, we remove them to maintain dataset integrity.

In [29]:
df = df.dropna(subset=['TotalCharges', 'Churn'])

In [30]:
df['Churn'].dtype
df['Churn'].head(10)

0     No
1     No
2    Yes
3     No
4    Yes
5    Yes
6     No
7     No
8    Yes
9     No
Name: Churn, dtype: object

In [31]:
df['Churn'].unique()

array(['No', 'Yes'], dtype=object)

### 🔁 Removing Duplicate Records

Ensuring each customer appears only once.

In [40]:
df = df.drop_duplicates()

### 🔄 Encoding Categorical Data

Convert the churn column into numerical format for analysis.

In [33]:
# Clean text
df['Churn'] = df['Churn'].astype(str).str.strip().str.lower()

# Create numeric version (DO NOT overwrite original)
df['Churn_numeric'] = df['Churn'].map({'yes': 1, 'no': 0})
df['Churn_numeric'] = df['Churn_numeric'].fillna(0).astype(int)

In [34]:

# Check AFTER mapping
df['Churn'].value_counts()

Churn
no     5163
yes    1869
Name: count, dtype: int64

### ⚙️ Feature Engineering

Creating a new metric: average monthly spend per customer.


In [35]:
df = df[df['tenure'] > 0]
df['AvgMonthlySpend'] = df['TotalCharges'] / df['tenure']

### ✅ Final Data Verification

Ensuring the dataset is clean and ready for analysis.

In [36]:
df.shape
df['Churn'].value_counts()

Churn
no     5163
yes    1869
Name: count, dtype: int64

In [37]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn_numeric,AvgMonthlySpend
count,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000
mean,0.162400,32.421786,64.798208,2283.300441,0.265785,64.799424
std,0.368844,24.545260,30.085974,2266.771362,0.441782,30.185891
min,0.000000,1.000000,18.250000,18.800000,0.000000,13.775000
25%,0.000000,9.000000,35.587500,401.450000,0.000000,36.179891
50%,0.000000,29.000000,70.350000,1397.475000,0.000000,70.373239
75%,0.000000,55.000000,89.862500,3794.737500,1.000000,90.179560
max,1.000000,72.000000,118.750000,8684.800000,1.000000,121.400000


#### 🎯 Summary of Data Cleaning

- Converted incorrect data types
- Handled missing values
- Removed duplicates
- Created a derived feature for deeper insights
##### 🚀 Outcome
The dataset is now clean, consistent, and ready for advanced analysis.
##### 💡 Key Insight
Data cleaning is a critical step in data science, as it directly impacts the quality and reliability of insights.

In [38]:
df.to_csv("/Users/hp/Documents/ASTU/CSEC_Projects/DataScience/telco-churn-analysis/data/raw/cleaned_telco.csv", index=False)

This is to save the cleaned dataset to ensure reproducibility and consistency across analysis stages